# GOLD ATP PLAYER-SEASON STATS

## Imports

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql.window import Window
import pandas as pd

import os
os.environ['SPARK_LOCAL_IP'] = '127.0.0.1'

from dotenv import load_dotenv
load_dotenv()

True

## Init spark

In [2]:
try:
    spark = SparkSession.builder.appName("fact_player_season_stats").getOrCreate()
except Exception as e:
    print(e)

In [3]:
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)

spark.conf.set("spark.sql.repl.eagerEval.maxNumRows", 200)
spark.conf.set("spark.sql.repl.eagerEval.truncate", 50)

## Load database

In [4]:
# # silver
# tb_player_match = (
#     spark.read
#     .format("jdbc")
#     .option("url", os.getenv("JDBC_URL"))
#     .option("dbtable", "silver.tb_atp_tournaments")
#     .option("user", os.getenv("DB_USER"))
#     .option("password", os.getenv("DB_PASSWORD"))
#     .option("driver", "org.postgresql.Driver")
#     .load()
#     )

# # gold
# tb_tournaments = (
#     spark.read
#     .format("jdbc")
#     .option("url", os.getenv("JDBC_URL"))
#     .option("dbtable", "gold.dim_tournaments")
#     .option("user", os.getenv("DB_USER"))
#     .option("password", os.getenv("DB_PASSWORD"))
#     .option("driver", "org.postgresql.Driver")
#     .load()
#     )

# tb_players = (
#     spark.read
#     .format("jdbc")
#     .option("url", os.getenv("JDBC_URL"))
#     .option("dbtable", "gold.dim_players")
#     .option("user", os.getenv("DB_USER"))
#     .option("password", os.getenv("DB_PASSWORD"))
#     .option("driver", "org.postgresql.Driver")
#     .load()
#     )

tb_player_match = spark.read.csv("../../../data/silver/tb_atp_player_match.csv", sep=',', header=True)
tb_tournaments = spark.read.csv("../../../data/gold/dimension/dim_tournaments.csv", sep=',', header=True)
tb_players = spark.read.csv("../../../data/gold/dimension/dim_players.csv", sep=',', header=True)

## Player-Season

In [5]:
round_order = (
    f.when(f.col("MATCH_ROUND") == "F", 11) # Final
     .when(f.col("MATCH_ROUND") == "SF", 10) # Semi Final
     .when(f.col("MATCH_ROUND") == "BR", 9) # Third Place
     .when(f.col("MATCH_ROUND") == "QF", 8) # Quarte final
     .when(f.col("MATCH_ROUND") == "R16", 7) # Round of 16
     .when(f.col("MATCH_ROUND") == "R32", 6) # Round of 32
     .when(f.col("MATCH_ROUND") == "R64", 5) # Round of 64
     .when(f.col("MATCH_ROUND") == "R128", 4) # Round of 128
     .when(f.col("MATCH_ROUND") == "ER", 3) # Early Round
     .when(f.col("MATCH_ROUND") == "RR", 2) # Round Robin
     .otherwise(1)
)

df = (
    tb_player_match.alias("p_m")
    .join(tb_tournaments.alias("t"), "TOURNEY_ID", 'left')
    .join(
        tb_players.alias("p"),
        (f.col("p_m.PLAYER_ID").cast("string") == f.col("p.PLAYER_ID").cast("string"))
        | (
            f.col("p_m.PLAYER_ID").cast("string")
            == f.col("p.PLAYER_ID_OLD").cast("string")
        ),
        "left",
    )
    .groupBy(f.col("SK_PLAYER"), f.substring(f.col("MATCH_DATE"), 1, 4).alias("REF_YEAR"))
    .agg(
        f.count("MATCH_ID").alias("TOTAL_MATCHES"),
        f.countDistinct("TOURNEY_ID").alias("TOTAL_TOURNAMENTS"),
        f.sum(f.when(f.col("PLAYER_IS_WINNER") == True, f.lit(1)).otherwise(f.lit(0))).alias("TOTAL_WINS"),
        f.sum(f.when(f.col("PLAYER_IS_WINNER") == False, f.lit(1)).otherwise(f.lit(0))).alias("TOTAL_LOSSES"),

        f.sum(
            f.when(
                (f.col("PLAYER_IS_WINNER") == True) & (f.col("MATCH_ROUND") == 'F'), f.lit(1)
                ).otherwise(0)
        ).alias("TOTAL_TITLES"),

        f.coalesce(f.max_by(
            f.when(f.col("TOURNEY_NAME") == "Australian Open", f.col("MATCH_ROUND")), 
            f.when(f.col("TOURNEY_NAME") == "Australian Open", round_order)
        ), f.lit('-')).alias("AUS_OPEN_RESULT"),
        
        f.coalesce(f.max_by(
            f.when(f.col("TOURNEY_NAME") == "Roland Garros", f.col("MATCH_ROUND")), 
            f.when(f.col("TOURNEY_NAME") == "Roland Garros", round_order)
        ), f.lit('-')).alias("ROLAND_GARROS_RESULT"),
        
        f.coalesce(f.max_by(
            f.when(f.col("TOURNEY_NAME") == "Wimbledon", f.col("MATCH_ROUND")), 
            f.when(f.col("TOURNEY_NAME") == "Wimbledon", round_order)
        ), f.lit('-')).alias("WIMBLEDON_RESULT"),
        
        f.coalesce(f.max_by(
            f.when(f.col("TOURNEY_NAME") == "US Open", f.col("MATCH_ROUND")), 
            f.when(f.col("TOURNEY_NAME") == "US Open", round_order)
        ), f.lit('-')).alias("US_OPEN_RESULT"),

        f.min_by(f.col("PLAYER_RANK"), f.col("MATCH_DATE")).cast('int').alias("PLAYER_START_RANK"),
        f.max_by(f.col("PLAYER_RANK"), f.col("MATCH_DATE")).cast('int').alias("PLAYER_FINAL_RANK"),
        f.min(f.col("PLAYER_RANK").cast('int')).alias("PLAYER_BEST_RANK"),
        f.max(f.col("PLAYER_RANK").cast('int')).alias("PLAYER_WORST_RANK"),

        f.min_by(f.col("PLAYER_RANK_PTS"), f.col("MATCH_DATE")).cast('int').alias("PLAYER_START_RANK_PTS"),
        f.max_by(f.col("PLAYER_RANK_PTS"), f.col("MATCH_DATE")).cast('int').alias("PLAYER_FINAL_RANK_PTS"),
        f.max(f.col("PLAYER_RANK_PTS").cast('int')).alias("PLAYER_BEST_RANK_PTS"),
        f.min(f.col("PLAYER_RANK_PTS").cast('int')).alias("PLAYER_WORST_RANK_PTS"),

        f.first("p_m.DATE_INGESTION").alias("DATE_INGESTION")
    )
    .orderBy("REF_YEAR")
)

In [6]:
if df.groupBy("SK_PLAYER", "REF_YEAR").agg(f.count("*").alias("total")).where("total > 1").count() == 0:
    print('ok')
else: 
    print('not ok')

ok


## Save dataframe

### Local

In [ ]:
df.toPandas().to_csv(
    r"../../../data/gold/fact/fact_player_season.csv",
    index=False,
    sep=",",
    encoding="utf-8"
)

### Supabase

In [8]:
(
df.write
    .format("jdbc")
    .option("url", os.getenv("JDBC_URL"))
    .option("dbtable", "gold.fact_player_tournament_stats")
    .option("user", os.getenv("DB_USER"))
    .option("password", os.getenv("DB_PASSWORD"))
    .option("driver", "org.postgresql.Driver")
    .mode("overwrite")
    .save()
)